# AdvisorX Strategy Backtest

Faithful backtest of the actual deployed CoinDCX bot logic (marketStateAnalyzer, strategyRouter, opportunityScorer, stopLossCalculator, takeProfitManagement - "balanced" preset), run against real historical CoinDCX data.

**Honest scope notes, read before trusting the numbers:**
- Data source: CoinDCX's own public `candles` API (free, no key needed) - the same exchange you actually trade on.
- **CoinDCX only allows 1m/15m/1h/1d directly for "B-" (futures-type) pairs.** This notebook fetches 1m candles and aggregates them up to 5m/15m/1h itself.
- **Performance bug found and fixed (3rd real run):** the backtest engine was silently O(n^2) - the confirm/filter timeframe slices grew unboundedly across the whole backtest instead of using a fixed rolling window, so every indicator got recomputed over an ever-larger window on every bar. This ran fine on short ranges but ground to a crawl (and disconnected the Colab runtime) on a full year of data. Fixed by bounding to a 200-candle rolling window (same approach already used for the primary timeframe) - benchmarked afterward at a consistent ~4ms/bar regardless of dataset size, projecting **~7-8 minutes for a full year's backtest** (separate from the 1m data fetch itself, which takes its own few minutes).
- **Bug found and fixed (2nd real run):** staged take-profit R-accounting was under-crediting profit on trades progressing through 2+ stages. Fixed and verified against a hand-calculated example.
- **Real limitation, not a bug:** the stop-loss check only tests each candle's CLOSE against the stop, not intra-bar high/low - a real stop order executes on any intrabar touch, so this backtest can UNDERSTATE real stop-loss losses.
- The reversal score does NOT include the MACD/RSI divergence component (10%+10% of the live bot's full weighting).
- **The live bot currently has NO time-based forced-close** (a real gap - the original engine's 36-hour rule was never ported). `max_hold_bars` below is a backtest-only knob for exploring this.
- Assumes 1% account risk per trade for the equity curve. One position at a time per symbol.
- Written and logic-tested with synthetic data (no live internet access in the environment that built this).


## Setup

In [ ]:
!pip install requests pandas numpy matplotlib -q

## Indicators (ported from indicators.js)

In [ ]:
"""
Faithful port of indicators.js - formulas copied from the deployed JS
source, not re-derived. Every function here mirrors its JS counterpart
exactly.
"""
import numpy as np
import pandas as pd


def ema(closes, period):
    """Standard EMA, seeded with an SMA of the first `period` values."""
    closes = np.asarray(closes, dtype=float)
    if len(closes) < period:
        return np.array([])
    out = np.zeros(len(closes) - period + 1)
    k = 2 / (period + 1)
    out[0] = closes[:period].mean()
    for i in range(1, len(out)):
        out[i] = closes[period - 1 + i] * k + out[i - 1] * (1 - k)
    return out


def rsi(closes, period=14):
    """Wilder-style RSI. Matches indicators.js: returns 100 if avgLoss==0
    (a real edge case on perfectly monotonic data, matches source exactly)."""
    closes = np.asarray(closes, dtype=float)
    if len(closes) < period + 1:
        return None
    deltas = np.diff(closes)
    gains = np.where(deltas > 0, deltas, 0)
    losses = np.where(deltas < 0, -deltas, 0)
    avg_gain = gains[:period].mean()
    avg_loss = losses[:period].mean()
    for i in range(period, len(deltas)):
        avg_gain = (avg_gain * (period - 1) + gains[i]) / period
        avg_loss = (avg_loss * (period - 1) + losses[i]) / period
    if avg_loss == 0:
        return 100.0
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def macd(closes, fast=12, slow=26, signal=9):
    """Returns dict with macd, signal, histogram (last value only)."""
    closes = np.asarray(closes, dtype=float)
    if len(closes) < slow + signal:
        return None
    ema_fast = ema(closes, fast)
    ema_slow = ema(closes, slow)
    # align lengths (ema_fast is longer since fast period < slow)
    offset = len(ema_fast) - len(ema_slow)
    macd_line = ema_fast[offset:] - ema_slow
    signal_line = ema(macd_line, signal)
    macd_val = macd_line[-1]
    signal_val = signal_line[-1]
    return {"macd": macd_val, "signal": signal_val, "histogram": macd_val - signal_val}


def macd_series(closes, fast=12, slow=26, signal=9):
    """Full histogram series (needed for macd_turn detection)."""
    closes = np.asarray(closes, dtype=float)
    if len(closes) < slow + signal:
        return np.array([])
    ema_fast = ema(closes, fast)
    ema_slow = ema(closes, slow)
    offset = len(ema_fast) - len(ema_slow)
    macd_line = ema_fast[offset:] - ema_slow
    signal_line = ema(macd_line, signal)
    off2 = len(macd_line) - len(signal_line)
    return macd_line[off2:] - signal_line


def macd_histogram_turn(closes):
    """1 = turned up, -1 = turned down, 0 = none. Matches indicators.js."""
    hist = macd_series(closes)
    if len(hist) < 3:
        return 0
    prev_prev, prev, latest = hist[-3], hist[-2], hist[-1]
    if prev_prev > prev and prev < latest and latest > 0:
        return 1
    if prev_prev < prev and prev > latest and latest < 0:
        return -1
    return 0


def atr_wilder(candles, period=14):
    """Wilder-smoothed ATR, matches stopLossCalculator.ts's calculateATR."""
    if len(candles) < period + 1:
        return 0
    highs = candles["high"].values
    lows = candles["low"].values
    closes = candles["close"].values
    trs = []
    for i in range(1, len(candles)):
        tr = max(
            highs[i] - lows[i],
            abs(highs[i] - closes[i - 1]),
            abs(lows[i] - closes[i - 1]),
        )
        trs.append(tr)
    trs = np.array(trs)
    atr = trs[:period].mean()
    for i in range(period, len(trs)):
        atr = (atr * (period - 1) + trs[i]) / period
    return atr


def atr_simple(candles, period=14):
    """Simple-average ATR, matches multiTimeframeAnalysis.ts's calculateATR
    (used for atr_ratio, distinct from the Wilder version used for stops)."""
    if len(candles) < period + 1:
        return 0
    highs = candles["high"].values
    lows = candles["low"].values
    closes = candles["close"].values
    trs = []
    for i in range(1, len(candles)):
        tr = max(
            highs[i] - lows[i],
            abs(highs[i] - closes[i - 1]),
            abs(lows[i] - closes[i - 1]),
        )
        trs.append(tr)
    trs = np.array(trs)
    if len(trs) < period:
        return 0
    return trs[-period:].mean()


def atr_ratio(candles, period=14):
    """Current ATR(14) vs ATR(14) from 20 candles ago."""
    current = atr_simple(candles, period)
    if len(candles) >= period + 20 + 1:
        historical = atr_simple(candles.iloc[:-20], period)
    else:
        historical = current
    return current / historical if historical != 0 else 1.0


def bollinger_bands(closes, period=20, std_dev=2):
    closes = np.asarray(closes, dtype=float)
    if len(closes) < period:
        return {"upper": 0, "middle": 0, "lower": 0}
    recent = closes[-period:]
    middle = recent.mean()
    std = recent.std()
    return {"upper": middle + std_dev * std, "middle": middle, "lower": middle - std_dev * std}


## Market state classifier (ported from marketStateAnalyzer.js)

In [ ]:
"""
Faithful port of marketStateAnalyzer.js. Every threshold/formula copied
directly from the deployed JS source.

SCOPE NOTE (flagged, not hidden): the reversal score's MACD/RSI divergence
components (10%+10% of the JS version's weighting) are NOT ported here -
reconstructing them for a fast vectorized backtest adds real complexity for
a relatively small slice of the score. The primary/confirm/filter
trend-weakening components (80% of the weighting) ARE ported in full. This
means reversal-triggered exits in this backtest are somewhat less sensitive
than the live bot - if this matters for your conclusions, ask and it can be
added.
"""
import numpy as np

OVERSOLD_EXTREME = 20
OVERSOLD_MILD = 30
OVERBOUGHT_EXTREME = 80
OVERBOUGHT_MILD = 70


def build_timeframe_indicators(candles):
    closes = candles["close"].values
    current_price = closes[-1]
    ema20 = ema(closes, 20)
    ema50 = ema(closes, 50)
    e20 = ema20[-1] if len(ema20) else 0
    e50 = ema50[-1] if len(ema50) else 0
    m = macd(closes) or {"macd": 0, "signal": 0, "histogram": 0}
    deviation = ((current_price - e20) / e20 * 100) if e20 != 0 else 0
    return {
        "current_price": current_price,
        "ema20": e20,
        "ema50": e50,
        "macd": m["macd"],
        "macd_signal": m["signal"],
        "macd_histogram": m["histogram"],
        "macd_turn": macd_histogram_turn(closes),
        "rsi7": rsi(closes, 7) if rsi(closes, 7) is not None else 50,
        "rsi14": rsi(closes, 14) if rsi(closes, 14) is not None else 50,
        "atr_ratio": atr_ratio(candles, 14),
        "deviation_from_ema20": deviation,
        "candles": candles,
    }


def determine_trend_strength(tf):
    if tf["ema20"] > tf["ema50"] and tf["macd"] > 0:
        return "trending_up"
    if tf["ema20"] < tf["ema50"] and tf["macd"] < 0:
        return "trending_down"
    return "ranging"


def determine_momentum_state(tf):
    if tf["rsi7"] < OVERSOLD_EXTREME:
        return "oversold_extreme"
    if tf["rsi7"] < OVERSOLD_MILD:
        return "oversold_mild"
    if tf["rsi7"] > OVERBOUGHT_EXTREME:
        return "overbought_extreme"
    if tf["rsi7"] > OVERBOUGHT_MILD:
        return "overbought_mild"
    return "neutral"


def determine_market_state(trend_strength, momentum_state, tf_confirm):
    state = "no_clear_signal"
    confidence = 0.3

    if trend_strength == "trending_up" and momentum_state == "oversold_extreme":
        state, confidence = "uptrend_oversold", 0.9
    elif trend_strength == "trending_down" and momentum_state == "overbought_extreme":
        state, confidence = "downtrend_overbought", 0.9
    elif trend_strength == "trending_down" and momentum_state == "oversold_extreme":
        state, confidence = "downtrend_oversold", 0.6
    elif trend_strength == "trending_up" and momentum_state == "overbought_extreme":
        state, confidence = "uptrend_overbought", 0.6
    elif trend_strength == "trending_up" and momentum_state in ("oversold_mild", "neutral"):
        state, confidence = "uptrend_continuation", 0.7
    elif trend_strength == "trending_down" and momentum_state in ("overbought_mild", "neutral"):
        state, confidence = "downtrend_continuation", 0.7
    elif trend_strength == "trending_down" and momentum_state == "oversold_mild":
        state, confidence = "downtrend_oversold", 0.5
    elif trend_strength == "trending_up" and momentum_state == "overbought_mild":
        state, confidence = "uptrend_overbought", 0.5
    elif trend_strength == "ranging" and momentum_state == "oversold_extreme":
        state, confidence = "ranging_oversold", 0.8
    elif trend_strength == "ranging" and momentum_state == "overbought_extreme":
        state, confidence = "ranging_overbought", 0.8
    elif trend_strength == "ranging" and momentum_state == "neutral":
        state, confidence = "ranging_neutral", 0.5

    if tf_confirm["macd_turn"] == 1 and state in ("uptrend_oversold", "ranging_oversold"):
        confidence = min(confidence + 0.1, 1.0)
    if tf_confirm["macd_turn"] == -1 and state in ("downtrend_overbought", "ranging_overbought"):
        confidence = min(confidence + 0.1, 1.0)

    return {"state": state, "confidence": confidence}


def calculate_trend_consistency(ema20a, ema50a, ema20b, ema50b, macd_a, macd_b):
    score = 0
    trend_a = 1 if ema20a > ema50a else -1
    momentum_a = 1 if macd_a > 0 else -1
    trend_b = 1 if ema20b > ema50b else -1
    momentum_b = 1 if macd_b > 0 else -1
    if trend_a == trend_b:
        score += 0.4
    if momentum_a == momentum_b:
        score += 0.3
    if trend_a == momentum_a:
        score += 0.15
    if trend_b == momentum_b:
        score += 0.15
    return max(0, min(1, score))


def calculate_triple_timeframe_consistency(tf_primary, tf_confirm, tf_filter):
    pc = calculate_trend_consistency(
        tf_primary["ema20"], tf_primary["ema50"], tf_confirm["ema20"], tf_confirm["ema50"],
        tf_primary["macd"], tf_confirm["macd"])
    cf = calculate_trend_consistency(
        tf_confirm["ema20"], tf_confirm["ema50"], tf_filter["ema20"], tf_filter["ema50"],
        tf_confirm["macd"], tf_filter["macd"])
    return pc * 0.6 + cf * 0.4


def calculate_trend_score(tf):
    """-100..100. Matches source exactly: EMA gap (40%) + MACD/price
    normalized (30%) + price deviation from EMA20 (20%) + RSI trend (10%)."""
    score = 0
    ema_gap = (tf["ema20"] - tf["ema50"]) / tf["ema50"] if tf["ema50"] != 0 else 0
    score += max(-40, min(40, ema_gap * 1000))
    macd_normalized = tf["macd"] / tf["current_price"] if tf["current_price"] != 0 else 0
    score += max(-30, min(30, macd_normalized * 10000))
    score += max(-20, min(20, tf["deviation_from_ema20"] * 2))
    rsi_trend = (tf["rsi7"] - 50) / 5
    score += max(-10, min(10, rsi_trend))
    return round(score)


def detect_trend_weakening(current_score, score_history):
    """Matches source exactly: 20%-relative-drop weakening, +/-20 crossing
    = reversing."""
    previous_score = score_history[-1] if len(score_history) > 0 else current_score
    is_weakening = abs(current_score) < abs(previous_score) * 0.8
    weakening_severity = (
        round((1 - abs(current_score) / abs(previous_score)) * 100) if is_weakening and previous_score != 0 else 0
    )
    return {"current_score": current_score, "previous_score": previous_score, "is_weakening": is_weakening,
            "weakening_severity": weakening_severity}


def calculate_reversal_score(tf_primary, tf_confirm, tf_filter, position_direction, history):
    """Weighted 40/25/15 across primary/confirm/filter timeframes (the
    trend-weakening portion of the source's reversal score - MACD/RSI
    divergence NOT included, see module docstring)."""
    score = 0
    details = []
    reversed_frames = []
    target_sign = -1 if position_direction == "long" else 1

    score_primary = calculate_trend_score(tf_primary)
    score_confirm = calculate_trend_score(tf_confirm)
    score_filter = calculate_trend_score(tf_filter)

    primary_change = detect_trend_weakening(score_primary, history.get("primary", []))
    if np.sign(score_primary) == target_sign and abs(score_primary) > 30:
        score += 40
        details.append(f"primary strongly reversed ({score_primary})")
        reversed_frames.append("primary")
    elif primary_change["is_weakening"] and primary_change["weakening_severity"] > 40:
        score += 20
        details.append(f"primary weakening ({primary_change['weakening_severity']}%)")
    elif abs(score_primary) < 20:
        score += 12

    confirm_change = detect_trend_weakening(score_confirm, history.get("confirm", []))
    if np.sign(score_confirm) == target_sign and abs(score_confirm) > 30:
        score += 25
        details.append(f"confirm strongly reversed ({score_confirm})")
        reversed_frames.append("confirm")
    elif confirm_change["is_weakening"] and confirm_change["weakening_severity"] > 40:
        score += 12

    filter_change = detect_trend_weakening(score_filter, history.get("filter", []))
    if np.sign(score_filter) == target_sign and abs(score_filter) > 30:
        score += 15
        details.append(f"filter reversed ({score_filter})")
        reversed_frames.append("filter")

    return {
        "reversal_score": score,
        "reversed_frames": reversed_frames,
        "details": details,
        "trend_scores": {"primary": score_primary, "confirm": score_confirm, "filter": score_filter},
    }


## Strategy logic (ported from strategyUtils.js / trendFollowingStrategy.js / meanReversionStrategy.js / breakoutStrategy.js / strategyRouter.js)

In [ ]:
"""
Faithful port of strategyUtils.js / trendFollowingStrategy.js /
meanReversionStrategy.js / breakoutStrategy.js / strategyRouter.js.

Matches what's actually DEPLOYED (not the untouched original): the
macdHist bug is fixed, breakout is wired in as a fallback-only extension
tagged is_breakout_extension.
"""
import numpy as np


def calculate_signal_strength(rsi7, macd_val, macd_signal, ema_alignment, price_position, trend_consistency):
    score = 0
    if rsi7 < 25:
        score += 25 * (25 - rsi7) / 25
    elif rsi7 > 75:
        score += 25 * (rsi7 - 75) / 25
    elif 30 <= rsi7 <= 70:
        score += 15

    macd_diff = macd_val - macd_signal
    if abs(macd_diff) > 0:
        score += 20 * min(abs(macd_diff) / 100, 1)

    if ema_alignment:
        score += 25

    abs_dev = abs(price_position)
    if abs_dev < 3:
        score += 15 * (1 - abs_dev / 3)

    score += 15 * trend_consistency
    return min(score / 100, 1)


def check_mtf_alignment(tf15m, tf1h, direction):
    alignment = 0
    ema_15 = tf15m["ema20"] > tf15m["ema50"]
    ema_1h = tf1h["ema20"] > tf1h["ema50"]
    if direction == "long" and ema_15 and ema_1h:
        alignment += 30
    elif direction == "short" and not ema_15 and not ema_1h:
        alignment += 30
    elif direction == "long" and ema_1h:
        alignment += 15
    elif direction == "short" and not ema_1h:
        alignment += 15

    macd_15 = tf15m["macd"] > 0
    macd_1h = tf1h["macd"] > 0
    if direction == "long" and macd_1h:
        alignment += 25
        if macd_15:
            alignment += 10
    elif direction == "short" and not macd_1h:
        alignment += 25
        if not macd_15:
            alignment += 10

    if direction == "long":
        if tf1h["rsi14"] < 70:
            alignment += 15
        if tf15m["rsi7"] < 30:
            alignment += 10
    else:
        if tf1h["rsi14"] > 30:
            alignment += 15
        if tf15m["rsi7"] > 70:
            alignment += 10

    if direction == "long" and tf1h["current_price"] > tf1h["ema20"]:
        alignment += 10
    elif direction == "short" and tf1h["current_price"] < tf1h["ema20"]:
        alignment += 10

    final_score = alignment / 100
    return {"aligned": final_score >= 0.6, "score": final_score}


def calculate_volatility_adjustment(atr, atr_ma=1.0):
    ratio = atr / atr_ma if atr_ma != 0 else 1
    if ratio < 0.8:
        return {"leverage_multiplier": 1.0, "status": "low"}
    if ratio < 1.2:
        return {"leverage_multiplier": 1.0, "status": "normal"}
    if ratio < 1.8:
        return {"leverage_multiplier": 0.8, "status": "high"}
    return {"leverage_multiplier": 0.6, "status": "extreme"}


def detect_macd_histogram_reversal(current_hist, previous_hist, direction):
    if direction == "bullish":
        return current_hist > previous_hist and previous_hist < 0
    return current_hist < previous_hist and previous_hist > 0


def identify_key_levels(candles, lookback=20):
    recent = candles.tail(lookback)
    return {"resistance": recent["high"].max(), "support": recent["low"].min()}


# ---- Trend-following ----

def trend_following_signal(symbol, direction, tf15m, tf1h, market_state):
    warnings = []
    if direction == "long":
        trend_confirmed = tf1h["ema20"] > tf1h["ema50"]
    else:
        trend_confirmed = tf1h["ema20"] < tf1h["ema50"]

    if not trend_confirmed:
        return {"action": "wait", "signal_strength": 0, "strategy_type": "trend_following"}

    signal_strength = 0
    if market_state["state"] in ("uptrend_continuation", "downtrend_continuation"):
        rsi_ok = (45 <= tf15m["rsi7"] <= 65) if direction == "long" else (35 <= tf15m["rsi7"] <= 55)
        if rsi_ok:
            signal_strength = 0.5
        else:
            return {"action": "wait", "signal_strength": 0, "strategy_type": "trend_following"}
    else:
        pullback_ok = tf15m["rsi7"] < 40 if direction == "long" else tf15m["rsi7"] > 60
        if not pullback_ok:
            return {"action": "wait", "signal_strength": 0, "strategy_type": "trend_following"}
        alignment = check_mtf_alignment(tf15m, tf1h, direction)
        price_pos = (tf15m["current_price"] - tf15m["ema20"]) / tf15m["ema20"] * 100
        signal_strength = calculate_signal_strength(
            tf15m["rsi7"], tf1h["macd"], tf1h["macd_signal"], trend_confirmed, price_pos, alignment["score"])

    vol_adj = calculate_volatility_adjustment(market_state["atr_ratio"], 1.0)
    if vol_adj["status"] == "extreme":
        signal_strength *= 0.7
    elif vol_adj["status"] == "high":
        signal_strength *= 0.85

    return {"action": direction, "signal_strength": signal_strength, "strategy_type": "trend_following"}


# ---- Mean-reversion (macdHist bug FIXED, matches deployed version) ----

def mean_reversion_signal(symbol, direction, tf15m, tf1h, market_state):
    extreme = tf15m["rsi7"] < 35 if direction == "long" else tf15m["rsi7"] > 65
    if not extreme:
        return {"action": "wait", "signal_strength": 0, "strategy_type": "mean_reversion"}

    if direction == "long":
        strong_opposite = tf1h["ema20"] < tf1h["ema50"] and tf1h["macd"] < -50
    else:
        strong_opposite = tf1h["ema20"] > tf1h["ema50"] and tf1h["macd"] > 50
    if strong_opposite:
        return {"action": "wait", "signal_strength": 0, "strategy_type": "mean_reversion"}

    alignment = check_mtf_alignment(tf15m, tf1h, direction)
    price_pos = (tf15m["current_price"] - tf15m["ema20"]) / tf15m["ema20"] * 100
    ema_align = tf1h["ema20"] > tf1h["ema50"] if direction == "long" else tf1h["ema20"] < tf1h["ema50"]
    signal_strength = calculate_signal_strength(
        tf15m["rsi7"], tf15m["macd"], tf15m["macd_signal"], ema_align, price_pos, alignment["score"] * 0.7)

    if direction == "long" and tf15m["rsi7"] < 25:
        signal_strength = min(signal_strength * 1.2, 1.0)
    elif direction == "short" and tf15m["rsi7"] > 75:
        signal_strength = min(signal_strength * 1.2, 1.0)

    # Fixed bug: consistent field name + real prevMacdHistogram (this
    # backtest computes it directly, matching the deployed fix)
    macd_reversal = detect_macd_histogram_reversal(
        tf15m["macd_histogram"], tf15m.get("prev_macd_histogram", tf15m["macd_histogram"]),
        "bullish" if direction == "long" else "bearish")
    if macd_reversal:
        signal_strength = min(signal_strength * 1.15, 1.0)

    vol_adj = calculate_volatility_adjustment(market_state["atr_ratio"], 1.0)
    if vol_adj["status"] == "extreme":
        signal_strength *= 0.6
    elif vol_adj["status"] == "high":
        signal_strength *= 0.8

    return {"action": direction, "signal_strength": signal_strength, "strategy_type": "mean_reversion"}


# ---- Breakout (flagged extension, matches deployed version - the
# original never actually calls this) ----

def breakout_signal(symbol, direction, tf15m, tf1h, market_state):
    candles = tf15m["candles"]
    if len(candles) < 20:
        return {"action": "wait", "signal_strength": 0, "strategy_type": "breakout"}

    levels = identify_key_levels(candles, 20)
    price = tf15m["current_price"]

    if direction == "long":
        broke = price > levels["resistance"] * 0.998
    else:
        broke = price < levels["support"] * 1.002
    if not broke:
        return {"action": "wait", "signal_strength": 0, "strategy_type": "breakout"}

    if direction == "long":
        rsi_ok = 35 <= tf15m["rsi7"] <= 75
    else:
        rsi_ok = 25 <= tf15m["rsi7"] <= 65
    if not rsi_ok:
        return {"action": "wait", "signal_strength": 0, "strategy_type": "breakout"}

    alignment = check_mtf_alignment(tf15m, tf1h, direction)
    ref_level = levels["resistance"] if direction == "long" else levels["support"]
    price_pos = (price - ref_level) / ref_level * 100
    ema_align = tf1h["ema20"] > tf1h["ema50"] if direction == "long" else tf1h["ema20"] < tf1h["ema50"]
    signal_strength = calculate_signal_strength(
        tf15m["rsi7"], tf1h["macd"], tf1h["macd_signal"], ema_align, price_pos, alignment["score"])

    vol_adj = calculate_volatility_adjustment(market_state["atr_ratio"], 1.0)
    if vol_adj["status"] == "extreme":
        signal_strength *= 0.7
    elif vol_adj["status"] == "high":
        signal_strength *= 0.85

    return {"action": direction, "signal_strength": signal_strength, "strategy_type": "breakout",
            "is_breakout_extension": True}


# ---- Router (matches strategyRouter.js's real state->strategy mapping) ----

STATE_STRATEGY_MAP = {
    "uptrend_oversold": ("trend_following", "long"),
    "downtrend_overbought": ("trend_following", "short"),
    "downtrend_oversold": ("mean_reversion", "long"),
    "uptrend_overbought": ("mean_reversion", "short"),
    "uptrend_continuation": ("trend_following", "long"),
    "downtrend_continuation": ("trend_following", "short"),
    "ranging_oversold": ("mean_reversion", "long"),
    "ranging_overbought": ("mean_reversion", "short"),
}


def route_strategy(symbol, market_state, tf15m, tf1h):
    state = market_state["state"]
    mapping = STATE_STRATEGY_MAP.get(state)

    if mapping:
        strategy_type, direction = mapping
        if strategy_type == "trend_following":
            base_result = trend_following_signal(symbol, direction, tf15m, tf1h, market_state)
        else:
            base_result = mean_reversion_signal(symbol, direction, tf15m, tf1h, market_state)
    else:
        base_result = {"action": "wait", "signal_strength": 0, "strategy_type": "none"}

    if base_result["action"] == "wait":
        long_breakout = breakout_signal(symbol, "long", tf15m, tf1h, market_state)
        if long_breakout["action"] == "long":
            return long_breakout
        short_breakout = breakout_signal(symbol, "short", tf15m, tf1h, market_state)
        if short_breakout["action"] == "short":
            return short_breakout

    return base_result


## Scoring, stop-loss, take-profit (ported from opportunityScorer.js / stopLossCalculator.js / takeProfitManagement.js)

In [ ]:
"""
Faithful port of opportunityScorer.js (balanced preset only, matching what's
deployed), stopLossCalculator.js, and takeProfitManagement.js's REAL
enforced logic (confirmed via the deployed code: always 1R/2R/3R at
33.33/33.33/0%, volatility-adjusted - NOT the per-strategy text description
that's never actually read).
"""
import numpy as np

# ---- Opportunity scorer (balanced preset) ----

BALANCED_WEIGHTS = {"signal_strength": 30, "trend_consistency": 25, "volatility_fit": 20,
                     "risk_reward": 15, "liquidity": 10, "min_score": 75}
BALANCED_VOL_PREFS = {"ideal_min": 0.8, "ideal_max": 1.2, "acceptable_min": 0.6,
                       "acceptable_max": 1.5, "penalty_factor": 0.5}

TIER1 = {"BTC", "ETH"}
TIER2 = {"BNB", "SOL", "XRP", "ADA"}
TIER3 = {"DOGE", "AVAX", "DOT", "MATIC", "LTC", "ARB", "OP"}


def calculate_volatility_fit_score(atr_ratio_val):
    p = BALANCED_VOL_PREFS
    if p["ideal_min"] <= atr_ratio_val <= p["ideal_max"]:
        return 1.0
    if p["acceptable_min"] <= atr_ratio_val <= p["acceptable_max"]:
        if atr_ratio_val < p["ideal_min"]:
            distance = p["ideal_min"] - atr_ratio_val
            rng = p["ideal_min"] - p["acceptable_min"]
        else:
            distance = atr_ratio_val - p["ideal_max"]
            rng = p["acceptable_max"] - p["ideal_max"]
        return 1.0 - (distance / rng) * p["penalty_factor"]
    return 0.3


def calculate_risk_reward_score(market_state_name):
    if market_state_name in ("uptrend_oversold", "downtrend_overbought"):
        return 0.9
    if market_state_name in ("uptrend_continuation", "downtrend_continuation"):
        return 0.7
    if market_state_name in ("ranging_oversold", "ranging_overbought"):
        return 0.8
    return 0.5


def calculate_liquidity_score(symbol):
    if symbol in TIER1:
        return 1.0
    if symbol in TIER2:
        return 0.85
    if symbol in TIER3:
        return 0.7
    return 0.6


def score_opportunity(strategy_result, market_state, alignment_score, atr_ratio_val, symbol):
    if strategy_result["action"] == "wait":
        return {"total_score": 0, "confidence": "low"}

    w = BALANCED_WEIGHTS
    signal_score = strategy_result["signal_strength"] * w["signal_strength"]
    trend_score = alignment_score * w["trend_consistency"]
    vol_score = calculate_volatility_fit_score(atr_ratio_val) * w["volatility_fit"]
    rr_score = calculate_risk_reward_score(market_state["state"]) * w["risk_reward"]
    liq_score = calculate_liquidity_score(symbol) * w["liquidity"]

    total = signal_score + trend_score + vol_score + rr_score + liq_score
    high_threshold = w["min_score"]
    medium_threshold = high_threshold - 15
    confidence = "high" if total >= high_threshold else ("medium" if total >= medium_threshold else "low")

    return {"total_score": round(total), "confidence": confidence}


# ---- Stop-loss calculator (hybrid ATR + support/resistance) ----

def find_support_level(candles, lookback=20):
    if len(candles) < lookback:
        return 0
    recent = candles.tail(lookback).reset_index(drop=True)
    lows = recent["low"].values
    local_lows = []
    for i in range(2, len(recent) - 2):
        if lows[i] < min(lows[i - 1], lows[i - 2]) and lows[i] < min(lows[i + 1], lows[i + 2]):
            local_lows.append(lows[i])
    return min(local_lows) if local_lows else lows.min()


def find_resistance_level(candles, lookback=20):
    if len(candles) < lookback:
        return 0
    recent = candles.tail(lookback).reset_index(drop=True)
    highs = recent["high"].values
    local_highs = []
    for i in range(2, len(recent) - 2):
        if highs[i] > max(highs[i - 1], highs[i - 2]) and highs[i] > max(highs[i + 1], highs[i + 2]):
            local_highs.append(highs[i])
    return max(local_highs) if local_highs else highs.max()


def calculate_scientific_stop_loss(candles, side, entry_price, atr_multiplier=2.0,
                                    min_stop_pct=0.5, max_stop_pct=5.0):
    current_price = candles["close"].iloc[-1]
    atr = atr_wilder(candles, 14)
    atr_pct = (atr / current_price) * 100 if current_price != 0 else 0
    atr_distance = atr * atr_multiplier
    atr_stop = entry_price - atr_distance if side == "long" else entry_price + atr_distance

    sr_stop = None
    if side == "long":
        support = find_support_level(candles, 20)
        buf = support * 0.001
        candidate = support - buf
        if candidate < entry_price:
            sr_stop = candidate
    else:
        resistance = find_resistance_level(candles, 20)
        buf = resistance * 0.001
        candidate = resistance + buf
        if candidate > entry_price:
            sr_stop = candidate

    if sr_stop is not None:
        final_stop = max(atr_stop, sr_stop) if side == "long" else min(atr_stop, sr_stop)
    else:
        final_stop = atr_stop

    if side == "long" and final_stop >= entry_price:
        final_stop = entry_price * (1 - min_stop_pct / 100)
    elif side == "short" and final_stop <= entry_price:
        final_stop = entry_price * (1 + min_stop_pct / 100)

    stop_distance_pct = (
        (entry_price - final_stop) / entry_price * 100 if side == "long"
        else (final_stop - entry_price) / entry_price * 100
    )

    quality_score = 50
    if 1.5 <= atr_pct <= 3.0:
        quality_score += 20
    elif atr_pct < 1.5:
        quality_score += 10
    if 1.5 <= stop_distance_pct <= 3.0:
        quality_score += 20
    elif stop_distance_pct < 1.5:
        quality_score += 10
    quality_score += 10  # S/R contribution (simplified: always some level found)
    quality_score = max(0, min(100, quality_score))

    return {
        "stop_price": final_stop,
        "stop_distance_pct": stop_distance_pct,
        "quality_score": quality_score,
        "volatility_extreme": atr_pct >= 5.0,
    }


def should_open_position(candles, side, entry_price, min_stop_pct=0.5, max_stop_pct=5.0, min_quality=40):
    result = calculate_scientific_stop_loss(candles, side, entry_price, 2.0, min_stop_pct, max_stop_pct)
    if result["stop_distance_pct"] < min_stop_pct:
        return False, result
    if result["stop_distance_pct"] > max_stop_pct:
        return False, result
    if result["volatility_extreme"]:
        return False, result
    if result["quality_score"] < min_quality:
        return False, result
    return True, result


# ---- Take-profit (real enforced logic: fixed 1R/2R/3R @ 33.33/33.33/0,
# volatility-adjusted - confirmed via grep that per-strategy text is never
# actually read by the enforcement code) ----

def analyze_market_volatility(candles):
    if len(candles) < 15:
        return {"level": "NORMAL", "adjustment_factor": 1.0}
    atr14 = atr_wilder(candles, 14)
    current_price = candles["close"].iloc[-1]
    atr_pct = (atr14 / current_price) * 100 if current_price != 0 else 3.0
    if atr_pct < 2:
        return {"level": "LOW", "adjustment_factor": 0.8}
    if atr_pct < 5:
        return {"level": "NORMAL", "adjustment_factor": 1.0}
    if atr_pct < 8:
        return {"level": "HIGH", "adjustment_factor": 1.2}
    return {"level": "EXTREME", "adjustment_factor": 1.5}


def calculate_r_multiple(entry_price, current_price, stop_loss_price, side):
    risk_distance = abs(entry_price - stop_loss_price)
    if risk_distance == 0:
        return 0
    profit_distance = (current_price - entry_price) if side == "long" else (entry_price - current_price)
    return profit_distance / risk_distance


def calculate_target_price(entry_price, stop_loss_price, r_multiple, side):
    risk_distance = abs(entry_price - stop_loss_price)
    target_distance = risk_distance * r_multiple
    return entry_price + target_distance if side == "long" else entry_price - target_distance


## CoinDCX historical data fetcher (public API, no key needed)

In [ ]:
"""
Fetches historical OHLCV data from CoinDCX's PUBLIC candles endpoint -
confirmed free, no API key required. Switched from Binance after a real
Binance API call from Colab returned HTTP 451 "Unavailable For Legal
Reasons" - this is a well-documented, longstanding restriction: Binance
blocks its main API from certain server regions, and Google Colab's
servers fall in that range (confirmed via multiple independent reports,
including one that literally says "Google Colab is not available for the
Binance API"). CoinDCX doesn't have this issue, and it's also more
representative since it's the actual exchange you trade on.

Confirmed directly from CoinDCX's own official docs
(https://docs.coindcx.com/): GET /market_data/candles supports pair,
interval, startTime, endTime (both in ms), and limit (max 1000) - this
genuinely supports historical range queries, not just "most recent N".

IMPORTANT DISCOVERY: CoinDCX's docs list 5m/15m/30m/1h/2h/4h/6h/8h/1d/... as
generally valid intervals for this endpoint, but that list is apparently
generic across pair types - "B-" prefixed pairs (futures/margin-type, which
is what B-BTC_USDT is) actually only support 1m/15m/1h/1d in practice
(confirmed via a real 422 error requesting 5m directly - this matches
EXACTLY the same restriction found while building the live bot's own
candle-fetching code). Fix is the same one already used there: fetch 1m
(which works) and aggregate up to whatever finer granularity is needed via
resample_candles() below - OHLCV aggregation composes correctly through
intermediate levels, so 1m->5m->15m gives identical results to 1m->15m
directly.

NOTE: this sandbox has no network access, so this still couldn't be
live-tested here either. Do a small test pull first in Colab before
running a long backtest, same caution as before.
"""
import time
import requests
import pandas as pd

BASE = "https://public.coindcx.com/market_data/candles"

INTERVAL_MS = {
    "1m": 60_000, "5m": 300_000, "15m": 900_000, "30m": 1_800_000,
    "1h": 3_600_000, "4h": 14_400_000, "1d": 86_400_000,
}


def fetch_coindcx_klines(symbol="BTC", interval="5m", start_time=None, end_time=None, limit_per_call=1000):
    """
    symbol: base symbol, e.g. "BTC" (builds the futures-style pair
    "B-{symbol}_USDT" - matches the live bot's own pair convention)
    interval: one of CoinDCX's documented intervals (1m, 5m, 15m, 30m, 1h,
    4h, 1d, ...)
    start_time / end_time: pandas.Timestamp-parseable or None (None
    end_time = now)
    Returns a DataFrame indexed by open_time, columns: open, high, low,
    close, volume
    """
    pair = f"B-{symbol}_USDT"
    if end_time is None:
        end_time = pd.Timestamp.utcnow()
    start_ms = int(pd.Timestamp(start_time).timestamp() * 1000)
    end_ms = int(pd.Timestamp(end_time).timestamp() * 1000)
    step_ms = INTERVAL_MS[interval] * limit_per_call

    all_rows = []
    cursor = start_ms
    request_count = 0
    start_wall_time = time.time()
    total_span_ms = end_ms - start_ms

    while cursor < end_ms:
        params = {
            "pair": pair,
            "interval": interval,
            "startTime": cursor,
            "endTime": min(cursor + step_ms, end_ms),
            "limit": limit_per_call,
        }
        resp = requests.get(BASE, params=params, timeout=15)
        resp.raise_for_status()
        rows = resp.json()
        request_count += 1

        if not rows:
            cursor += step_ms
            time.sleep(0.3)
            continue
        all_rows.extend(rows)
        cursor = max(r["time"] for r in rows) + INTERVAL_MS[interval]
        time.sleep(0.3)  # polite pacing

        # Progress every 20 requests - so a genuinely slow-but-working fetch
        # is visibly distinguishable from one that's actually stuck. If you
        # don't see this print advancing every few seconds, something's
        # actually wrong (network issue, rate limiting) rather than just slow.
        if request_count % 20 == 0:
            pct_done = min(100, round((cursor - start_ms) / total_span_ms * 100)) if total_span_ms > 0 else 100
            elapsed = time.time() - start_wall_time
            print(f"  ...{request_count} requests done, ~{pct_done}% through the date range, "
                  f"{len(all_rows)} candles so far, {elapsed:.0f}s elapsed")

    if not all_rows:
        raise ValueError(
            f"No candles returned for {pair} {interval} in this range - double check the pair "
            f"exists on CoinDCX and the date range isn't before it was listed."
        )

    df = pd.DataFrame(all_rows)
    df["open_time"] = pd.to_datetime(df["time"], unit="ms")
    df = df.set_index("open_time")[["open", "high", "low", "close", "volume"]].sort_index()
    return df[~df.index.duplicated(keep="first")]


def resample_candles(base_candles, target_interval):
    """Resamples finer candles up to a coarser interval - e.g. 5m -> 1h -
    same aggregation logic as the live bot's aggregateCandles (first open,
    last close, max high, min low, summed volume), using pandas resample."""
    rule_map = {"5m": "5min", "15m": "15min", "1h": "1h", "4h": "4h", "1d": "1D"}
    rule = rule_map[target_interval]
    out = base_candles.resample(rule).agg({
        "open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum",
    })
    return out.dropna()


## Backtest engine

In [ ]:
"""
The backtest engine. Walks forward through 5m candles (the "primary"
timeframe for the balanced preset), building 15m ("confirm") and 1h
("filter") context at each step from ONLY fully-closed higher-timeframe
candles as of that moment - this is the standard way to avoid look-ahead
bias when backtesting a multi-timeframe strategy: a partially-formed 15m/1h
candle must never leak information from bars that haven't happened yet in
the simulation.
"""
import pandas as pd
import numpy as np


MIN_CANDLES_NEEDED = 55  # matches the live bot's own minimum


def _closed_bucket_candles(resampled, as_of_time, rule_minutes, window=200):
    """Returns the last `window` fully-closed buckets as of `as_of_time` -
    bounded rolling window, not everything since the start of the dataset.

    PERFORMANCE BUG FIX: this used to do `resampled.loc[:as_of_time]` with
    no bound, which returns MORE data on every single iteration as the
    backtest progresses - every indicator (EMA/RSI/MACD) then gets
    recomputed from scratch over an ever-growing window, turning the whole
    backtest into an O(n^2) operation. For a short test range this is
    barely noticeable; over a full year (100k+ bars) it starts fast and
    grinds to a crawl by the end - this is almost certainly what caused
    the Colab runtime to hang/disconnect on the full-year run. Bounding to
    a fixed window (200 candles - plenty for EMA50/RSI14/MACD26, which are
    the longest lookbacks used) makes each iteration constant-time, so the
    whole backtest is O(n) instead of O(n^2).
    """
    sliced = resampled.loc[:as_of_time].tail(window)
    if len(sliced) == 0:
        return sliced
    last_bucket_start = sliced.index[-1]
    bucket_end = last_bucket_start + pd.Timedelta(minutes=rule_minutes)
    if bucket_end > as_of_time + pd.Timedelta(minutes=5):  # +5m: primary bar just closed at as_of_time
        return sliced.iloc[:-1]
    return sliced


def run_backtest(symbol, candles_5m, min_score=75, max_positions=1,
                  max_hold_bars=None, verbose=False):
    """
    candles_5m: DataFrame of 5-minute OHLCV candles (from fetch_binance_klines
    or equivalent), indexed by open_time, ascending.
    max_hold_bars: if set, force-closes a position after this many 5m bars
    (the missing 36-hour-style safety net discussed - None means no cap,
    matching what's currently deployed).

    Returns (trades_df, equity_curve_series).
    """
    candles_15m_full = resample_candles(candles_5m, "15m")
    candles_1h_full = resample_candles(candles_5m, "1h")

    trades = []
    open_position = None  # dict or None
    trend_history = {"primary": [], "confirm": [], "filter": []}
    equity = 1.0
    equity_curve = []

    n = len(candles_5m)
    for i in range(MIN_CANDLES_NEEDED, n):
        t = candles_5m.index[i]
        primary_slice = candles_5m.iloc[max(0, i - 200):i + 1]  # rolling window, enough history for EMA50 etc.
        confirm_slice = _closed_bucket_candles(candles_15m_full, t, 15)
        filter_slice = _closed_bucket_candles(candles_1h_full, t, 60)

        if len(primary_slice) < MIN_CANDLES_NEEDED or len(confirm_slice) < MIN_CANDLES_NEEDED or len(filter_slice) < MIN_CANDLES_NEEDED:
            equity_curve.append((t, equity))
            continue

        current_price = primary_slice["close"].iloc[-1]

        tf_primary = build_timeframe_indicators(primary_slice)
        tf_confirm = build_timeframe_indicators(confirm_slice)
        tf_filter = build_timeframe_indicators(filter_slice)

        # ---- Manage an existing position first ----
        if open_position is not None:
            pos = open_position
            direction = pos["direction"]

            reversal = calculate_reversal_score(tf_primary, tf_confirm, tf_filter, direction, trend_history)
            hit_stop = current_price <= pos["stop"] if direction == "long" else current_price >= pos["stop"]

            exit_reason = None
            exit_price = current_price
            if reversal["reversal_score"] >= 70:
                exit_reason = "reversal"
            elif hit_stop:
                exit_reason = "stop"
                exit_price = pos["stop"]
            elif max_hold_bars is not None and (i - pos["entry_index"]) >= max_hold_bars:
                exit_reason = "max_hold_time"
            else:
                # Staged take-profit check (real enforced logic: fixed
                # 1R/2R/3R, volatility-adjusted)
                vol = analyze_market_volatility(confirm_slice)  # 15m, matches deployed logic
                current_r = calculate_r_multiple(pos["entry"], current_price, pos["stop"], direction)
                adj_r = {k: v * vol["adjustment_factor"] for k, v in {"1": 1, "2": 2, "3": 3}.items()}
                if current_r >= adj_r["3"] and pos["stages_done"] == 2:
                    pos["stages_done"] = 3
                    # Stage 3 has closePercent=0 (matches the real deployed
                    # logic exactly) - nothing additional realized here,
                    # the remaining 33.34% just continues trailing.
                    pos["stop"] = calculate_target_price(pos["entry"], pos["stop"], 2, direction)  # trail to 2R
                elif current_r >= adj_r["2"] and pos["stages_done"] == 1:
                    pos["stages_done"] = 2
                    # BUG FIX: this closed slice realizes 33.33% of the
                    # position AT 2R, full stop - not the delta since stage
                    # 1. Each stage's contribution is independent
                    # (fraction closed x R achieved at that price), not a
                    # running subtraction from the previous stage.
                    pos["realized_r"] += adj_r["2"] * 0.3333
                    pos["last_staged_r"] = adj_r["2"]
                    pos["stop"] = calculate_target_price(pos["entry"], pos["stop"], 1, direction)  # trail to 1R
                elif current_r >= adj_r["1"] and pos["stages_done"] == 0:
                    pos["stages_done"] = 1
                    pos["realized_r"] += adj_r["1"] * 0.3333
                    pos["last_staged_r"] = adj_r["1"]
                    pos["stop"] = pos["entry"]  # move to breakeven

            if exit_reason:
                final_r = calculate_r_multiple(pos["entry"], exit_price, pos["initial_stop"], direction)
                # Blend already-realized partial R with the final remaining-size exit R
                remaining_fraction = 1.0 - (0.3333 * pos["stages_done"] if pos["stages_done"] < 3 else 0.6667)
                total_r = pos["realized_r"] + final_r * remaining_fraction
                trades.append({
                    "symbol": symbol, "direction": direction, "entry_time": pos["entry_time"], "exit_time": t,
                    "entry_price": pos["entry"], "exit_price": exit_price, "r_achieved": total_r,
                    "exit_reason": exit_reason, "setup_type": pos["setup_type"],
                    "is_breakout_extension": pos.get("is_breakout_extension", False),
                    "bars_held": i - pos["entry_index"],
                })
                equity *= (1 + total_r * 0.01)  # 1% risk per trade convention
                open_position = None

        # ---- Look for a new entry if flat ----
        elif open_position is None:
            trend_strength = determine_trend_strength(tf_primary)
            momentum_state = determine_momentum_state(tf_confirm)
            market_state = determine_market_state(trend_strength, momentum_state, tf_confirm)
            market_state["atr_ratio"] = tf_filter["atr_ratio"]

            alignment_score = calculate_triple_timeframe_consistency(tf_primary, tf_confirm, tf_filter)
            strategy_result = route_strategy(symbol, market_state, tf_confirm, tf_filter)
            opp = score_opportunity(strategy_result, market_state, alignment_score, tf_filter["atr_ratio"], symbol)

            if strategy_result["action"] != "wait" and opp["total_score"] >= min_score:
                direction = strategy_result["action"]
                can_open, sl_result = should_open_position(tf_filter["candles"], direction, current_price)
                if can_open:
                    open_position = {
                        "direction": direction, "entry": current_price, "entry_time": t, "entry_index": i,
                        "stop": sl_result["stop_price"], "initial_stop": sl_result["stop_price"],
                        "stages_done": 0, "realized_r": 0.0, "last_staged_r": 0.0,
                        "setup_type": strategy_result["strategy_type"],
                        "is_breakout_extension": strategy_result.get("is_breakout_extension", False),
                    }

        # Update trend-score history (for next iteration's reversal check)
        for key, tf in (("primary", tf_primary), ("confirm", tf_confirm), ("filter", tf_filter)):
            trend_history[key].append(calculate_trend_score(tf))
            if len(trend_history[key]) > 5:
                trend_history[key].pop(0)

        equity_curve.append((t, equity))

    trades_df = pd.DataFrame(trades)
    equity_series = pd.Series(dict(equity_curve))
    return trades_df, equity_series


def summarize_results(trades_df):
    if len(trades_df) == 0:
        return {"total_trades": 0}
    wins = trades_df[trades_df["r_achieved"] > 0]
    losses = trades_df[trades_df["r_achieved"] <= 0]
    return {
        "total_trades": len(trades_df),
        "win_rate_pct": round(len(wins) / len(trades_df) * 100, 1),
        "avg_r_all_trades": round(trades_df["r_achieved"].mean(), 3),
        "avg_r_wins": round(wins["r_achieved"].mean(), 3) if len(wins) else 0,
        "avg_r_losses": round(losses["r_achieved"].mean(), 3) if len(losses) else 0,
        "total_r": round(trades_df["r_achieved"].sum(), 2),
        "avg_bars_held": round(trades_df["bars_held"].mean(), 1),
        "exit_reason_breakdown": trades_df["exit_reason"].value_counts().to_dict(),
        "breakout_extension_trades": int(trades_df["is_breakout_extension"].sum()),
    }


## Run it

Fetches a full year of real 1m data for BTC from CoinDCX, aggregates it to 5m, runs the backtest, and shows results.

**Expect the data fetch to take a few minutes (you'll see progress lines every 20 requests), and the backtest itself to take roughly 7-8 minutes after that (now that the performance bug is fixed) - so budget 10-15 minutes total for a full year. This is normal, not a hang - if you don't see any output at all advancing for several minutes, that's when something's actually wrong.**

In [ ]:
SYMBOL = "BTC"                    # will fetch pair "B-BTCUSDT" from CoinDCX
START_DATE = "2024-01-01"
END_DATE = "2025-01-01"           # one full year
MIN_SCORE = 75                    # matches the live "balanced" preset threshold
MAX_HOLD_BARS = 96                # 96 x 5min = 8 hours; set to None to match the live bot's current (gap-flagged) behavior of no cap
RISK_PER_TRADE_PCT = 1.0          # used only for the equity curve, doesn't affect trade selection

print(f"Fetching {SYMBOL} 1m candles from CoinDCX ({START_DATE} to {END_DATE}) - this will take a few minutes...")
candles_1m = fetch_coindcx_klines(SYMBOL, "1m", START_DATE, END_DATE)
print(f"Fetched {len(candles_1m)} 1m candles. Aggregating to 5m...")
candles_5m = resample_candles(candles_1m, "5m")
print(f"Got {len(candles_5m)} 5m candles.")


In [ ]:
import time
print("Running backtest - this takes roughly 7-8 minutes for a full year, please wait...")
_start = time.time()
trades, equity = run_backtest(SYMBOL, candles_5m, min_score=MIN_SCORE, max_hold_bars=MAX_HOLD_BARS)
print(f"Backtest complete in {(time.time()-_start)/60:.1f} minutes.")
print(f"\nTotal trades: {len(trades)}")
if len(trades) > 0:
    summary = summarize_results(trades)
    for k, v in summary.items():
        print(f"{k}: {v}")


### Trade log

In [ ]:
trades.sort_values("entry_time") if len(trades) > 0 else trades

### Equity curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(equity.index, equity.values)
plt.title(f"{SYMBOL} - Equity Curve ({RISK_PER_TRADE_PCT}% risk per trade)")
plt.xlabel("Time")
plt.ylabel("Equity (starting = 1.0)")
plt.grid(alpha=0.3)
plt.show()


### Try other coins or hold-time caps

- `MAX_HOLD_BARS = None` vs a real cap - see how much the missing time-based exit actually matters
- ETH, SOL, DOGE, XRP - same logic, different coin's real history
- Different sub-ranges within the year - does the result hold up consistently, or is it concentrated in a specific stretch (e.g. the Jan 2024 ETF approval volatility)?
